In [30]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit

In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [4]:
df = pd.read_csv("data/datav3.csv")
df.head()

,customer_id,observation_month,branch_id,age,monthly_income,annual_income,website_visits_30d,website_visits_90d,session_minutes,discount_pct,support_tickets_30d,premium_member,days_since_last_purchase,cart_items,email_opens_30d,profile_views_30d,social_referrals_30d,survey_score,preferred_contact_hour,device_type,region,app_theme,education_level,marketing_channel,campaign_code,noise_feature_1,noise_feature_2,noise_feature_3,purchased
0,CUST01723,2026-06-01,B24,57,4144.46,49861.80,8,25,18.74,9.01,3,1,16.1,2,11,2,1,5,17,Desktop,West,Dark,Bachelor,Organic,CMP60,-0.3758,0.7415,1,0
1,CUST03090,2025-11-01,B05,39,3313.13,39964.60,7,27,12.38,7.64,1,1,6.6,0,11,6,1,7,8,Mobile,Central,System,Bachelor,Email,CMP02,-0.6708,0.5766,1,0
2,CUST04684,2026-01-01,B13,51,4965.57,58703.65,3,15,8.97,15.84,3,0,55.0,0,3,5,1,4,8,Mobile,North,Light,Master,Organic,CMP04,0.2868,-0.6751,0,0
3,CUST01687,2025-03-01,B05,56,2045.66,24364.68,7,24,30.73,10.50,0,0,22.8,0,6,8,0,5,8,Mobile,Central,Dark,Bachelor,Social,CMP10,2.1992,0.8292,1,0
4,CUST00535,2025-11-01,B17,31,4355.62,51882.27,10,29,31.40,2.09,0,0,40.1,1,10,4,0,8,8,Desktop,South,Dark,Bachelor,Search,CMP01,-1.2592,-0.1662,1,0


In [5]:
df.shape

(7519, 29)

# Assumptions

## Independence of Observations

### Repeated entities and Duplicated IDs

#### Duplicated IDs

In [6]:
df.duplicated().sum()

np.int64(35)

In [7]:
pd.set_option('display.max_rows', None)

In [8]:
exact_duplicates = df[df.duplicated(keep=False)]
exact_duplicates.sort_values(['customer_id', 'observation_month']).head(20)

,customer_id,observation_month,branch_id,age,monthly_income,annual_income,website_visits_30d,website_visits_90d,session_minutes,discount_pct,support_tickets_30d,premium_member,days_since_last_purchase,cart_items,email_opens_30d,profile_views_30d,social_referrals_30d,survey_score,preferred_contact_hour,device_type,region,app_theme,education_level,marketing_channel,campaign_code,noise_feature_1,noise_feature_2,noise_feature_3,purchased
5077,CUST00501,2025-08-01,B08,33,3078.55,36733.47,4,14,33.36,1.40,2,0,5.8,2,10,2,0,5,13,Mobile,East,Dark,Bachelor,Organic,CMP04,-0.3185,-0.9865,0,0
6080,CUST00501,2025-08-01,B08,33,3078.55,36733.47,4,14,33.36,1.40,2,0,5.8,2,10,2,0,5,13,Mobile,East,Dark,Bachelor,Organic,CMP04,-0.3185,-0.9865,0,0
731,CUST00605,2025-08-01,B13,62,5851.43,70200.80,11,40,17.22,7.82,2,0,19.5,1,13,5,1,6,8,Mobile,North,Dark,PhD,Organic,CMP02,-0.3049,-0.4817,1,0
999,CUST00605,2025-08-01,B13,62,5851.43,70200.80,11,40,17.22,7.82,2,0,19.5,1,13,5,1,6,8,Mobile,North,Dark,PhD,Organic,CMP02,-0.3049,-0.4817,1,0
5180,CUST00634,2026-01-01,B04,49,5360.73,64293.39,3,7,4.85,13.97,1,1,24.1,2,9,3,1,6,19,Mobile,Central,Dark,Master,Affiliate,CMP01,0.5543,0.4336,0,0
5830,CUST00634,2026-01-01,B04,49,5360.73,64293.39,3,7,4.85,13.97,1,1,24.1,2,9,3,1,6,19,Mobile,Central,Dark,Master,Affiliate,CMP01,0.5543,0.4336,0,0
2884,CUST00639,2026-01-01,B14,36,3496.38,42021.33,7,22,26.89,21.72,0,0,27.7,3,4,4,1,5,14,Mobile,North,Dark,High School,Referral,CMP01,-0.0494,0.0916,0,0
6333,CUST00639,2026-01-01,B14,36,3496.38,42021.33,7,22,26.89,21.72,0,0,27.7,3,4,4,1,5,14,Mobile,North,Dark,High School,Referral,CMP01,-0.0494,0.0916,0,0
1601,CUST00887,2026-02-01,B15,38,4849.39,58041.66,10,30,20.81,21.70,1,0,32.0,1,9,6,1,6,19,Mobile,North,Dark,Bachelor,Social,CMP02,0.5303,-0.0009,0,0
6975,CUST00887,2026-02-01,B15,38,4849.39,58041.66,10,30,20.81,21.70,1,0,32.0,1,9,6,1,6,19,Mobile,North,Dark,Bachelor,Social,CMP02,0.5303,-0.0009,0,0


they are exact duplicated columns, I will drop them

In [9]:
df_clean = df.drop_duplicates()

In [10]:
df_clean.shape

(7484, 29)

#### Repeated entities

In [11]:
customer_counts = df_clean['customer_id'].value_counts()
customer_counts.head()

customer_id
CUST03090    3
CUST02133    3
CUST01676    3
CUST03203    3
CUST02897    3
Name: count, dtype: int64

In [12]:
repeated_customers = customer_counts[customer_counts > 1]
len(repeated_customers)

1775

In [13]:
pd.set_option('display.max_rows', None)

In [14]:
repeated_customers_df =  df_clean[df_clean['customer_id'].isin(repeated_customers.index)].sort_values(by='customer_id')
repeated_customers_df.head(10)

,customer_id,observation_month,branch_id,age,monthly_income,annual_income,website_visits_30d,website_visits_90d,session_minutes,discount_pct,support_tickets_30d,premium_member,days_since_last_purchase,cart_items,email_opens_30d,profile_views_30d,social_referrals_30d,survey_score,preferred_contact_hour,device_type,region,app_theme,education_level,marketing_channel,campaign_code,noise_feature_1,noise_feature_2,noise_feature_3,purchased
630,CUST00001,2025-12-01,B17,40,14331.28,172408.24,6,21,32.13,10.34,1,0,8.2,2,12,4,1,5,8,Mobile,South,Dark,Master,Search,CMP01,0.4028,-0.3411,1,0
4953,CUST00001,2025-11-01,B17,40,14525.17,173291.39,9,31,23.24,10.14,1,0,47.9,1,13,4,3,7,18,Mobile,South,Dark,Master,Partner_Rare,CMP02,1.4404,0.7305,1,0
4195,CUST00003,2026-04-01,B15,31,2283.80,27365.68,7,24,31.83,11.54,1,0,44.1,2,9,6,1,6,11,Mobile,North,Light,High School,Social,CMP06,2.5045,-0.0049,0,0
6624,CUST00003,2026-05-01,B15,31,2359.29,28074.30,7,23,33.04,7.09,2,0,50.2,0,5,3,0,5,19,Mobile,North,Light,High School,Search,CMP01,-0.2593,-0.6711,1,0
3236,CUST00004,2026-02-01,B15,27,4735.69,56815.43,9,33,43.15,23.18,0,0,15.4,3,13,8,1,6,11,Mobile,North,Dark,Bachelor,Referral,CMP01,1.1225,-0.0007,0,0
5128,CUST00004,2026-03-01,B15,27,4576.81,54304.18,6,22,17.01,16.45,1,0,19.2,2,3,2,1,7,8,Mobile,North,Dark,Bachelor,Search,CMP33,0.7423,0.2380,0,0
7101,CUST00006,2025-06-01,B10,40,6764.86,80682.02,11,33,25.74,28.96,1,0,25.9,4,10,4,3,9,9,Desktop,East,Light,Bachelor,Social,CMP01,0.5939,0.2385,1,0
2215,CUST00006,2025-07-01,B10,40,6702.88,80338.78,8,31,25.39,27.59,0,0,35.9,4,12,4,0,3,12,Desktop,East,Light,Bachelor,Social,CMP03,-0.9011,-0.9781,0,0
3514,CUST00006,2025-05-01,B10,40,6758.65,81038.19,12,39,30.81,7.78,0,0,45.7,2,16,5,0,7,21,Desktop,East,Light,Bachelor,Search,CMP02,-0.4524,0.1382,0,0
1474,CUST00007,2026-02-01,B14,36,4494.10,54132.37,10,32,29.26,2.86,0,0,6.6,2,9,6,0,5,13,Mobile,North,Light,PhD,Affiliate,CMP08,-0.1623,0.3194,0,0


In [15]:
customer_counts.value_counts().sort_values()

count
3     509
2    1266
1    3425
Name: count, dtype: int64

In [16]:
df_clean[
    df_clean['customer_id'] == 'CUST00006'
][
    [
        'customer_id',
        'observation_month',
        'monthly_income',
        'purchased'
    ]
].sort_values('observation_month')

,customer_id,observation_month,monthly_income,purchased
3514,CUST00006,2025-05-01,6758.65,0
7101,CUST00006,2025-06-01,6764.86,0
2215,CUST00006,2025-07-01,6702.88,0


they are not duplicated rows, they are legitimate repeated measurements.Same customers with different time points.                                         
Treatments:
- `Agrregation` if data has multiple lower-level rows, but we want unit of analysis should be higher-level.
- `Group-aware train/test splitting`

##### Aggregation

In [17]:
df_clean.head()

,customer_id,observation_month,branch_id,age,monthly_income,annual_income,website_visits_30d,website_visits_90d,session_minutes,discount_pct,support_tickets_30d,premium_member,days_since_last_purchase,cart_items,email_opens_30d,profile_views_30d,social_referrals_30d,survey_score,preferred_contact_hour,device_type,region,app_theme,education_level,marketing_channel,campaign_code,noise_feature_1,noise_feature_2,noise_feature_3,purchased
0,CUST01723,2026-06-01,B24,57,4144.46,49861.80,8,25,18.74,9.01,3,1,16.1,2,11,2,1,5,17,Desktop,West,Dark,Bachelor,Organic,CMP60,-0.3758,0.7415,1,0
1,CUST03090,2025-11-01,B05,39,3313.13,39964.60,7,27,12.38,7.64,1,1,6.6,0,11,6,1,7,8,Mobile,Central,System,Bachelor,Email,CMP02,-0.6708,0.5766,1,0
2,CUST04684,2026-01-01,B13,51,4965.57,58703.65,3,15,8.97,15.84,3,0,55.0,0,3,5,1,4,8,Mobile,North,Light,Master,Organic,CMP04,0.2868,-0.6751,0,0
3,CUST01687,2025-03-01,B05,56,2045.66,24364.68,7,24,30.73,10.50,0,0,22.8,0,6,8,0,5,8,Mobile,Central,Dark,Bachelor,Social,CMP10,2.1992,0.8292,1,0
4,CUST00535,2025-11-01,B17,31,4355.62,51882.27,10,29,31.40,2.09,0,0,40.1,1,10,4,0,8,8,Desktop,South,Dark,Bachelor,Search,CMP01,-1.2592,-0.1662,1,0


In [22]:
yearly_df = df_clean.copy()

yearly_df['observation_month'] = pd.to_datetime(yearly_df['observation_month'])
yearly_df['year'] = yearly_df['observation_month'].dt.year
yearly_df.head()

,customer_id,observation_month,branch_id,age,monthly_income,annual_income,website_visits_30d,website_visits_90d,session_minutes,discount_pct,support_tickets_30d,premium_member,days_since_last_purchase,cart_items,email_opens_30d,profile_views_30d,social_referrals_30d,survey_score,preferred_contact_hour,device_type,region,app_theme,education_level,marketing_channel,campaign_code,noise_feature_1,noise_feature_2,noise_feature_3,purchased,year
0,CUST01723,2026-06-01,B24,57,4144.46,49861.80,8,25,18.74,9.01,3,1,16.1,2,11,2,1,5,17,Desktop,West,Dark,Bachelor,Organic,CMP60,-0.3758,0.7415,1,0,2026
1,CUST03090,2025-11-01,B05,39,3313.13,39964.60,7,27,12.38,7.64,1,1,6.6,0,11,6,1,7,8,Mobile,Central,System,Bachelor,Email,CMP02,-0.6708,0.5766,1,0,2025
2,CUST04684,2026-01-01,B13,51,4965.57,58703.65,3,15,8.97,15.84,3,0,55.0,0,3,5,1,4,8,Mobile,North,Light,Master,Organic,CMP04,0.2868,-0.6751,0,0,2026
3,CUST01687,2025-03-01,B05,56,2045.66,24364.68,7,24,30.73,10.50,0,0,22.8,0,6,8,0,5,8,Mobile,Central,Dark,Bachelor,Social,CMP10,2.1992,0.8292,1,0,2025
4,CUST00535,2025-11-01,B17,31,4355.62,51882.27,10,29,31.40,2.09,0,0,40.1,1,10,4,0,8,8,Desktop,South,Dark,Bachelor,Search,CMP01,-1.2592,-0.1662,1,0,2025


In [25]:
customer_year_df = (
    yearly_df.groupby(
        ['customer_id', 'year']
    )
    .agg(
        monthly_income=('monthly_income', 'mean'),
        annual_income=('annual_income', 'mean'),

        website_visits_30d=('website_visits_30d', 'sum'),
        website_visits_90d=('website_visits_90d', 'sum'),

        session_minutes=('session_minutes', 'sum'),

        discount_pct=('discount_pct', 'mean'),

        support_tickets_30d=('support_tickets_30d', 'sum'),

        days_since_last_purchase=(
            'days_since_last_purchase',
            'last'
        ),

        cart_items=('cart_items', 'sum'),

        email_opens_30d=('email_opens_30d', 'sum'),

        profile_views_30d=(
            'profile_views_30d',
            'sum'
        ),

        social_referrals_30d=(
            'social_referrals_30d',
            'sum'
        ),

        survey_score=('survey_score', 'mean'),

        purchased=('purchased', 'max')
    )
    .reset_index()
)

In [26]:
customer_year_df.head()

,customer_id,year,monthly_income,annual_income,website_visits_30d,website_visits_90d,session_minutes,discount_pct,support_tickets_30d,days_since_last_purchase,cart_items,email_opens_30d,profile_views_30d,social_referrals_30d,survey_score,purchased
0,CUST00001,2025,14428.225,172849.815,15,52,55.37,10.240,2,47.9,3,25,8,4,6.0,0
1,CUST00002,2025,4940.630,59718.200,6,23,20.98,22.280,0,26.6,0,4,1,2,4.0,1
2,CUST00003,2026,2321.545,27719.990,14,47,64.87,9.315,3,50.2,2,14,9,1,5.5,0
3,CUST00004,2026,4656.250,55559.805,15,55,60.16,19.815,1,19.2,5,16,10,2,6.5,0
4,CUST00005,2025,4199.810,50368.140,12,40,15.96,12.420,0,60.0,3,17,9,1,5.0,0


##### Group-aware train-test split

In [28]:
df_clean.head()

,customer_id,observation_month,branch_id,age,monthly_income,annual_income,website_visits_30d,website_visits_90d,session_minutes,discount_pct,support_tickets_30d,premium_member,days_since_last_purchase,cart_items,email_opens_30d,profile_views_30d,social_referrals_30d,survey_score,preferred_contact_hour,device_type,region,app_theme,education_level,marketing_channel,campaign_code,noise_feature_1,noise_feature_2,noise_feature_3,purchased
0,CUST01723,2026-06-01,B24,57,4144.46,49861.80,8,25,18.74,9.01,3,1,16.1,2,11,2,1,5,17,Desktop,West,Dark,Bachelor,Organic,CMP60,-0.3758,0.7415,1,0
1,CUST03090,2025-11-01,B05,39,3313.13,39964.60,7,27,12.38,7.64,1,1,6.6,0,11,6,1,7,8,Mobile,Central,System,Bachelor,Email,CMP02,-0.6708,0.5766,1,0
2,CUST04684,2026-01-01,B13,51,4965.57,58703.65,3,15,8.97,15.84,3,0,55.0,0,3,5,1,4,8,Mobile,North,Light,Master,Organic,CMP04,0.2868,-0.6751,0,0
3,CUST01687,2025-03-01,B05,56,2045.66,24364.68,7,24,30.73,10.50,0,0,22.8,0,6,8,0,5,8,Mobile,Central,Dark,Bachelor,Social,CMP10,2.1992,0.8292,1,0
4,CUST00535,2025-11-01,B17,31,4355.62,51882.27,10,29,31.40,2.09,0,0,40.1,1,10,4,0,8,8,Desktop,South,Dark,Bachelor,Search,CMP01,-1.2592,-0.1662,1,0


In [31]:
X = df_clean.drop(columns=['purchased'])
y = df_clean['purchased']

groups = df_clean['customer_id']

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)


X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]



In [32]:
X_train.head()

,customer_id,observation_month,branch_id,age,monthly_income,annual_income,website_visits_30d,website_visits_90d,session_minutes,discount_pct,support_tickets_30d,premium_member,days_since_last_purchase,cart_items,email_opens_30d,profile_views_30d,social_referrals_30d,survey_score,preferred_contact_hour,device_type,region,app_theme,education_level,marketing_channel,campaign_code,noise_feature_1,noise_feature_2,noise_feature_3
0,CUST01723,2026-06-01,B24,57,4144.46,49861.80,8,25,18.74,9.01,3,1,16.1,2,11,2,1,5,17,Desktop,West,Dark,Bachelor,Organic,CMP60,-0.3758,0.7415,1
1,CUST03090,2025-11-01,B05,39,3313.13,39964.60,7,27,12.38,7.64,1,1,6.6,0,11,6,1,7,8,Mobile,Central,System,Bachelor,Email,CMP02,-0.6708,0.5766,1
2,CUST04684,2026-01-01,B13,51,4965.57,58703.65,3,15,8.97,15.84,3,0,55.0,0,3,5,1,4,8,Mobile,North,Light,Master,Organic,CMP04,0.2868,-0.6751,0
3,CUST01687,2025-03-01,B05,56,2045.66,24364.68,7,24,30.73,10.50,0,0,22.8,0,6,8,0,5,8,Mobile,Central,Dark,Bachelor,Social,CMP10,2.1992,0.8292,1
6,CUST03223,2026-02-01,B11,34,3717.06,44096.22,12,36,26.89,16.75,1,1,17.3,4,5,9,1,6,20,Mobile,North,Dark,Bachelor,Social,CMP01,1.1912,-0.4454,0


In [33]:
train_customers = set(df_clean.iloc[train_idx]['customer_id'])
test_customers = set(df_clean.iloc[test_idx]['customer_id'])

overlap = train_customers.intersection(test_customers)

print(overlap)

set()


### Natural groups or clusters

In [34]:
df_clean.head()

,customer_id,observation_month,branch_id,age,monthly_income,annual_income,website_visits_30d,website_visits_90d,session_minutes,discount_pct,support_tickets_30d,premium_member,days_since_last_purchase,cart_items,email_opens_30d,profile_views_30d,social_referrals_30d,survey_score,preferred_contact_hour,device_type,region,app_theme,education_level,marketing_channel,campaign_code,noise_feature_1,noise_feature_2,noise_feature_3,purchased
0,CUST01723,2026-06-01,B24,57,4144.46,49861.80,8,25,18.74,9.01,3,1,16.1,2,11,2,1,5,17,Desktop,West,Dark,Bachelor,Organic,CMP60,-0.3758,0.7415,1,0
1,CUST03090,2025-11-01,B05,39,3313.13,39964.60,7,27,12.38,7.64,1,1,6.6,0,11,6,1,7,8,Mobile,Central,System,Bachelor,Email,CMP02,-0.6708,0.5766,1,0
2,CUST04684,2026-01-01,B13,51,4965.57,58703.65,3,15,8.97,15.84,3,0,55.0,0,3,5,1,4,8,Mobile,North,Light,Master,Organic,CMP04,0.2868,-0.6751,0,0
3,CUST01687,2025-03-01,B05,56,2045.66,24364.68,7,24,30.73,10.50,0,0,22.8,0,6,8,0,5,8,Mobile,Central,Dark,Bachelor,Social,CMP10,2.1992,0.8292,1,0
4,CUST00535,2025-11-01,B17,31,4355.62,51882.27,10,29,31.40,2.09,0,0,40.1,1,10,4,0,8,8,Desktop,South,Dark,Bachelor,Search,CMP01,-1.2592,-0.1662,1,0


`1. Does the dataset have a meaningful grouping structure ?`

In [45]:
df_clean['branch_id'].nunique()

25

In [42]:
df_clean['branch_id'].value_counts().sort_values()

branch_id
B01    235
B05    252
B03    259
B15    264
B04    265
B02    268
B23    280
B25    282
B12    283
B22    293
B24    294
B09    296
B10    297
B14    297
B13    298
B11    302
B21    313
B08    316
B16    321
B19    321
B07    326
B17    343
B18    350
B06    353
B20    376
Name: count, dtype: int64

In [41]:
# number of unique customers in each branch
df.groupby('branch_id')['customer_id'].nunique().sort_values()

branch_id
B01    158
B04    173
B05    174
B03    177
B02    179
B15    189
B25    194
B23    196
B12    199
B10    203
B14    203
B11    203
B24    204
B22    210
B13    211
B09    211
B07    218
B19    219
B08    226
B21    227
B16    231
B06    241
B18    245
B17    247
B20    262
Name: customer_id, dtype: int64

the dataset has a meaningful grouping structure because multiple different customers belong to the same branch.

`2. Are observations within a group more similar than observations across groups?`

In [54]:
branch_summary = (
    df.groupby('branch_id')
      .agg(
          customers=('customer_id', 'nunique'),
          observations=('customer_id', 'size'),
          purchase_rate=('purchased', 'mean')
      )
)

branch_summary.sort_values(by='purchase_rate')

,customers,observations,purchase_rate
branch_id,,,
B11,203,303,0.075908
B17,247,344,0.110465
B14,203,299,0.120401
B09,211,298,0.120805
B18,245,352,0.122159
B10,203,299,0.127090
B15,189,267,0.127341
B12,199,284,0.130282
B04,173,267,0.138577


In [52]:
overall_purchase_rate = df['purchased'].mean()

overall_purchase_rate

np.float64(0.14509908232477722)

In [53]:
branch_rates = (
    df.groupby('branch_id')['purchased']
      .mean()
)

print("Min:", branch_rates.min())
print("Max:", branch_rates.max())
print("Std:", branch_rates.std())

Min: 0.07590759075907591
Max: 0.2361963190184049
Std: 0.029242373704262154


In [65]:
braches_stats = df_clean.groupby('branch_id')[
    ['age',
    'monthly_income', 
    'annual_income', 
    'website_visits_30d',
    'website_visits_90d', 
    'session_minutes', 
    'discount_pct',
    'support_tickets_30d', 
    'premium_member', 
    'days_since_last_purchase',
    'cart_items', 
    'email_opens_30d', 
    'profile_views_30d',
    'social_referrals_30d', 
    'survey_score', 
    'preferred_contact_hour',
    'noise_feature_1',
    'noise_feature_2', 
    'noise_feature_3', 
    'purchased']
].mean()

In [66]:
df_stats = df_clean[
    [
        'age',
        'monthly_income', 
        'annual_income', 
        'website_visits_30d',
        'website_visits_90d', 
        'session_minutes', 
        'discount_pct',
        'support_tickets_30d', 
        'premium_member', 
        'days_since_last_purchase',
        'cart_items', 
        'email_opens_30d', 
        'profile_views_30d',
        'social_referrals_30d', 
        'survey_score', 
        'preferred_contact_hour',
        'noise_feature_1',
        'noise_feature_2', 
        'noise_feature_3', 
        'purchased'
    ]
].mean()


In [63]:
comparison_df = braches_stats.copy()
comparison_df.loc['overall_data'] = df_stats

In [64]:
comparison_df

,age,monthly_income,annual_income,website_visits_30d,website_visits_90d,session_minutes,discount_pct,support_tickets_30d,premium_member,days_since_last_purchase,cart_items,email_opens_30d,profile_views_30d,social_referrals_30d,survey_score,preferred_contact_hour,noise_feature_1,noise_feature_2,noise_feature_3,purchased
branch_id,,,,,,,,,,,,,,,,,,,,
B01,39.855319,5153.626213,61882.214298,7.851064,27.008511,21.443617,15.244638,0.731915,0.246809,38.020000,2.451064,9.085106,5.195745,1.310638,6.025532,15.421277,0.023607,-0.037803,0.510638,0.161702
B02,39.645522,5255.355112,63078.280709,7.936567,27.130597,20.868694,14.541306,0.839552,0.238806,37.321269,2.238806,8.996269,5.279851,1.205224,6.026119,15.328358,-0.012708,-0.028828,0.518657,0.141791
B03,41.146718,5589.874981,67060.601892,8.038610,27.486486,22.278880,15.066139,0.691120,0.351351,36.696525,2.552124,9.776062,5.100386,1.258687,6.057915,15.953668,0.078197,-0.005442,0.563707,0.162162
B04,39.169811,5092.008340,61126.968038,7.924528,27.079245,21.434377,15.006113,0.641509,0.267925,37.894340,2.384906,9.426415,5.354717,1.411321,5.939623,15.150943,0.008302,-0.005658,0.505660,0.139623
B05,39.539683,5457.503413,65454.897937,8.019841,27.563492,20.525516,15.074127,0.829365,0.253968,38.037698,2.567460,9.051587,5.396825,1.293651,5.988095,14.996032,-0.035949,-0.010597,0.511905,0.178571
B06,39.192635,5406.285779,64902.669547,7.903683,26.991501,21.730170,14.545552,0.796034,0.249292,37.288669,2.311615,9.546742,5.093484,1.311615,5.957507,14.920680,0.008227,-0.029516,0.501416,0.152975
B07,39.334356,5378.191994,64577.192730,8.024540,27.438650,21.629969,14.557239,0.812883,0.334356,35.618712,2.481595,9.389571,5.309816,1.340491,5.957055,15.273006,0.063428,0.020986,0.521472,0.236196
B08,38.936709,5665.908544,67987.409937,8.370253,28.382911,21.341519,14.467500,0.740506,0.319620,37.746203,2.455696,9.664557,5.582278,1.351266,6.022152,14.632911,-0.064384,-0.009787,0.537975,0.142405
B09,37.902027,5353.416689,64264.046182,7.908784,26.929054,21.541858,14.916520,0.824324,0.216216,38.088514,2.293919,9.138514,5.354730,1.337838,5.814189,14.293919,-0.053457,0.025434,0.425676,0.121622
